# MutantHunter GRPO — 30-step Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jester1177/MetaOpenEnv_MutantHunter/blob/main/training/train_grpo.ipynb)

> **This notebook is a 30-step demo run for verification.** The full 200-step training
> results are at **[HF Hub URL placeholder]** and the W&B run is at **[W&B URL placeholder]**.
> To reproduce the full run on A100, change `STEPS` below to `200` (~4h).

It exercises the same code path as `training/train_grpo.py` end-to-end: env reset,
prompt build, policy generate, env step, GRPO update — just with a tiny step count
so a Colab free T4 can finish in 15–30 minutes.

## Configuration

In [ ]:
STEPS = 30                # demo run; set to 200 for the full A100 run
ROLLOUTS_PER_STEP = 2
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_NEW_TOKENS = 1024
LEARNING_RATE = 5e-6
SEED = 42
OUTPUT_DIR = "/content/grpo_demo"
WANDB_PROJECT = None       # set to a string to log; None disables W&B

## Install dependencies
On Colab the environment ships PyTorch + CUDA already; we add the training extras and bitsandbytes for 4-bit.

In [ ]:
%%bash
set -e
if [ ! -d /content/MetaOpenEnv_MutantHunter ]; then
  git clone https://github.com/jester1177/MetaOpenEnv_MutantHunter.git /content/MetaOpenEnv_MutantHunter
fi
cd /content/MetaOpenEnv_MutantHunter
pip install --quiet -e ".[training]"
pip install --quiet bitsandbytes wandb

## Authenticate (optional)
Replace the placeholders with your tokens to push artifacts / log to W&B. Skip this cell if you only want a local demo.

In [ ]:
import os

HF_TOKEN = "hf_PLACEHOLDER"          # https://huggingface.co/settings/tokens
WANDB_API_KEY = "PLACEHOLDER"        # https://wandb.ai/authorize

if HF_TOKEN and not HF_TOKEN.endswith("PLACEHOLDER"):
    from huggingface_hub import login
    login(token=HF_TOKEN)

if WANDB_API_KEY and not WANDB_API_KEY.endswith("PLACEHOLDER"):
    os.environ["HOME"] = "/tmp"      # avoid netrc-perm errors on shared runners
    import wandb
    wandb.login(key=WANDB_API_KEY)

## Imports

In [ ]:
import sys
from pathlib import Path

REPO = Path("/content/MetaOpenEnv_MutantHunter")
for p in (REPO, REPO / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from mutant_hunter.server.mutant_hunter_environment import MutantHunterEnvironment
from training.train_grpo import TrainingConfig, run, build_prompt

## Instantiate the env
`MutantHunterEnvironment` is the in-process OpenEnv environment. Each `reset(seed)` picks one
(repo, module) episode from the corpus manifest, copies the repo into a sandbox workspace,
loads the precomputed mutation baseline, and returns an `Observation` carrying the module
summary, existing-test names, and remaining tool-call budget.

In [ ]:
env = MutantHunterEnvironment()
obs = env.reset(seed=0)
print("repo:", obs.repo_name)
print("module:", obs.module_path)
print("baseline mutation score:", obs.baseline_mutation_score)
print("summary preview:", (obs.module_summary or "")[:300])
env.close()

## Reward function
Per submitted test file, the env runs three checks and composes a scalar reward via `mutant_hunter.rubric.compose_reward`:

| Component | What it measures |
|---|---|
| `mutation_kill` | fraction of baseline-surviving mutants killed by the new tests |
| `coverage_delta` | line-coverage improvement over the existing suite |
| `format` | parses + no forbidden imports/syscalls |
| `parsimony` | small reward for short, focused tests |
| `no_regression_gate` | hard gate: 0 if existing tests fail under the new suite |

GRPO sees only the final scalar reward — the breakdown is for diagnostics.

## Train (30 steps)
This calls the same `run()` entrypoint as `training/train_grpo.py`. No logic is duplicated — the notebook
just builds a `TrainingConfig` and hands it off.

In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
cfg = TrainingConfig(
    base_model=BASE_MODEL,
    steps=STEPS,
    rollouts_per_step=ROLLOUTS_PER_STEP,
    learning_rate=LEARNING_RATE,
    use_unsloth=False,         # T4 is happier on plain transformers + peft
    use_4bit=True,
    use_lora=True,
    lora_rank=16,
    max_new_tokens=MAX_NEW_TOKENS,
    seed=SEED,
    output_dir=Path(OUTPUT_DIR),
    wandb_project=WANDB_PROJECT,
)
run(cfg)

## Tiny eval (3 episodes, mutation_aware heuristic)
Quick sanity check: the heuristic baseline policy from `training/baseline_eval.py` runs against the
same env and prints a mean reward. This is the fast (~1 min) version of the bar-chart "heuristic" reference.

In [ ]:
from evaluation.eval_harness import run_local
from training.baseline_eval import mutation_aware_policy

report = run_local(mutation_aware_policy, n_episodes=3, seed_start=0)
print(f"\nmutation_aware mean_reward={report.mean_reward:.4f} "
      f"median_reward={report.median_reward:.4f}")

## Done

30 steps is far too few for the policy to actually learn — this notebook only verifies the loop
executes end-to-end on a free T4. For the trained results referenced in the README:

- Final LoRA adapter on HF Hub: **[HF Hub URL placeholder]**
- W&B run with full reward curves: **[W&B URL placeholder]**
- Plots: see `plots/` in the model repo